# Jet EPub Parser

Intructions: Collect as many epub versions of Jet Magazine as you can and put them in the `jet_epubs` directory.

Then run this script.

Note that each time you run this script, the DOC and LIB tables will be overwritten -- if you make changes to them and leave them in this directory, then all of your changes will be lost after running this script. So make sure to copy the files to another directory is you want to make changes to them.

## Set Up

In [122]:
epub_dir = "jet_epubs"

In [123]:
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
import pandas as pd
import glob

## Process

For each epub in the directory, convert the contents of the files into a LIB and DOC table.

The DOC table has a string for each magazine page.

In [124]:
doc_list = []
for epub_path in glob.glob("jet_epubs/*.epub"):
    
    # Create an epub object to work with
    book = epub.read_epub(epub_path)


    # COLLECT METADATA FOR LIB TABLE
    epub_id = epub_path.split("_")[-1].split(".")[0]
    date = book.get_metadata('DC', 'date')[0][0] if book.get_metadata('DC', 'date') else "Unknown"
    lib_list.append(dict(epub_id=epub_id, date=date))

    # COLLECT CONTENT FOR DOC TABLE
    # Initialize a list to store the "chapter" content of the epub
    chapters = []

    # Iterate over items in the epub file
    for item in book.get_items():
        
        # Look for document items (HTML/XML)
        if item.get_type() == ebooklib.ITEM_DOCUMENT:
        
            # Parse HTML content
            soup = BeautifulSoup(item.get_content(), 'html.parser')
            text = soup.get_text()
            chapters.append(text)

    # Create DataFrame
    # doc_list.append(pd.DataFrame(dict(epub_id=epub_id, chapter=chapters), columns=['doc_str']))
    doc_list.append((epub_id, date, chapters))

## Create LIB 

In [125]:
LIB = pd.DataFrame(doc_list)
LIB.columns = ['epub_id', 'date', 'pages']
LIB.index.name = 'mag_id'
LIB

,epub_id,date,pages
mag_id,,,
0,vo8DAAAAMBAJ,1972-12-07,[\n\n\n\nThis book was produced in EPUB format...
1,ProDAAAAMBAJ,1967-05-11,[\n\n\n\nThis book was produced in EPUB format...
2,dEIDAAAAMBAJ,1975-11-20,[\n\n\n\nThis book was produced in EPUB format...
3,YEIDAAAAMBAJ,1978-05-04,[\n\n\n\nThis book was produced in EPUB format...
4,AMADAAAAMBAJ,1978-07-06,[\n\n\n\nThis book was produced in EPUB format...
5,FDsDAAAAMBAJ,Unknown,[\n\n\n\n\n\n\n\n\n\n]
6,vq8DAAAAMBAJ,1960-03-10,[\n\n\n\nThis book was produced in EPUB format...
7,fbcDAAAAMBAJ,1958-02-20,[\n\n\n\nThis book was produced in EPUB format...
8,YkEDAAAAMBAJ,1959-08-20,[\n\n\n\nThis book was produced in EPUB format...


## Extract DOC from LIB

In [126]:
DOC = LIB.pages.apply(pd.Series).stack().to_frame('doc_str')
DOC.index.names = LIB.index.names + ['page_num']
DOC

doc_str
mag_id page_num                                                   
0      0         \n\n\n\nThis book was produced in EPUB format ...
       1         \n\n\n\nThe text on this page is estimated to ...
       2         \n\n\n\nWarning: The Surgeon General Has Deter...
       3         \n\n\n\nEditor and Publisher John H. Johnson E...
       4         \n\n\n\nREADERS RAP A1 Green’s Home Discussed ...
...                                                            ...
9      65        \n\n\n\n■ ■■■ MOVIE OF THE WEEK ■■■■ THE NAKED...
       66        \n\n\n\nJtADIO-IV Sammy Davis Jr., Johnny Math...
       67        \n\n\n\nPROUD TO BE A MOTHER By LENA HORNE Len...
       68        \n\n\n\nMAKE MONEY I IN YOUR OWN BUSINESS! No ...
       69                    \n\n\n\n\nJet\n\n\nNotice\n\n\n\n\n\n

[631 rows x 1 columns]

## Remove cruft from DOC strings 

In [128]:
DOC.doc_str = DOC.doc_str.str.strip()
DOC

doc_str
mag_id page_num                                                   
0      0         This book was produced in EPUB format by the I...
       1         The text on this page is estimated to be only ...
       2         Warning: The Surgeon General Has Determined Th...
       3         Editor and Publisher John H. Johnson Executive...
       4         READERS RAP A1 Green’s Home Discussed Dear Edi...
...                                                            ...
9      65        ■ ■■■ MOVIE OF THE WEEK ■■■■ THE NAKED MAJA On...
       66        JtADIO-IV Sammy Davis Jr., Johnny Mathis On 'O...
       67        PROUD TO BE A MOTHER By LENA HORNE Lena Home, ...
       68        MAKE MONEY I IN YOUR OWN BUSINESS! No Investme...
       69                                          Jet\n\n\nNotice

[631 rows x 1 columns]

In [129]:
## Save
DOC.to_csv("jet-DOC.csv", index=True)
LIB.iloc[:,:-1].to_csv("jet-LIB.csv", index=True)